In [ ]:
# Лабораторная работа 5.2
# Генерация текста с помощью LSTM (символьный уровень)

# Задачи:
# 1. Загрузить текстовый корпус.
# 2. Создать алфавит и преобразовать текст в числовые последовательности.
# 3. Реализовать модель LSTM для предсказания следующего символа.
# 4. Обучить модель на последовательностях символов.
# 5. Реализовать функцию генерации текста с температурой.
# 6. Проанализировать качество сгенерированных текстов.


# ============================================================
# ИМПОРТ БИБЛИОТЕК
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Проверка CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")


In [ ]:
# ============================================================
# 1. ЗАГРУЗКА ТЕКСТОВОГО КОРПУСА
# ============================================================

# ЗАДАНИЕ: Загрузите текстовый файл
# - Отрывок из "Войны и мира" (war_and_peace.txt)

filepath = "war_and_peace.txt"  # ЗАМЕНИТЕ НА СВОЙ ПУТЬ

# ЗАДАНИЕ: Реализуйте загрузку текста из файла
# Используйте open() с кодировкой utf-8
# Если файл не найден, используйте синтетический текст
text = ""  # <-- ДОПОЛНИТЕ КОД
print(f"Загружен текст. Длина: {len(text)} символов")

# Вывод первых 500 символов для ознакомления
print("\nПервые 500 символов текста:")
print("-" * 50)
print(text[:500])
print("-" * 50)


In [ ]:
# ============================================================
# 2. СОЗДАНИЕ АЛФАВИТА И КОДИРОВАНИЕ ТЕКСТА
# ============================================================

# ЗАДАНИЕ: Создайте алфавит (уникальные символы)
# chars = sorted(list(set(text)))
chars = ...  # <-- ДОПОЛНИТЕ КОД
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)

print(f"Размер алфавита: {vocab_size}")
print(f"Первые 20 символов алфавита: {''.join(chars[:20])}")

# Пример кодирования и декодирования
sample_text = "Привет, мир!"
encoded = [char_to_idx[ch] for ch in sample_text if ch in char_to_idx]
decoded = ''.join([idx_to_char[idx] for idx in encoded])
print(f"Исходный текст: {sample_text}")
print(f"Закодированный: {encoded}")
print(f"Декодированный: {decoded}")

In [ ]:
# ============================================================
# 3. ФОРМИРОВАНИЕ ПОСЛЕДОВАТЕЛЬНОСТЕЙ ДЛЯ ОБУЧЕНИЯ
# ============================================================

# ЗАДАНИЕ: Реализуйте функцию create_sequences
def create_sequences(text, seq_length):
    """
    Создание последовательностей для обучения.
    
    Параметры:
    - text: строка с текстом
    - seq_length: длина входной последовательности
    
    Возвращает:
    - inputs: массив входных последовательностей (индексы символов)
    - targets: массив целевых символов (следующий символ)
    """
    inputs, targets = [], []
    # ЗАДАНИЕ: Реализуйте цикл
    # for i in range(len(text) - seq_length):
    #     inputs.append([...])
    #     targets.append([...])
    return np.array(inputs), np.array(targets)

# Параметры последовательности
seq_length = 100
print(f"Длина последовательности: {seq_length}")

# Создание последовательностей
X, y = create_sequences(text, seq_length)
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Пример входа (первые 10 индексов): {X[0][:10]}")
print(f"Пример цели: {y[0]} -> символ '{idx_to_char[y[0]]}'")

# Преобразование в тензоры PyTorch
X_t = torch.LongTensor(X).to(device)
y_t = torch.LongTensor(y).to(device)

# Создание DataLoader
batch_size = 128
dataset = TensorDataset(X_t, y_t)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
print(f"Количество батчей: {len(train_loader)}")

In [ ]:
# ============================================================
# 4. ОПРЕДЕЛЕНИЕ МОДЕЛИ LSTM
# ============================================================

class CharLSTM(nn.Module):
    """
    Модель LSTM для генерации текста на уровне символов.
    """
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(CharLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # ЗАДАНИЕ: Определите слои модели
        # self.embedding = nn.Embedding(...)
        # self.lstm = nn.LSTM(...)
        # self.fc = nn.Linear(...)
        
    def forward(self, x, hidden=None):
        """
        Прямой проход модели.
        
        Параметры:
        - x: входные данные (batch_size, seq_length)
        - hidden: скрытое состояние
        
        Возвращает:
        - out: логиты (batch_size, seq_length, vocab_size)
        - hidden: обновлённое скрытое состояние
        """
        # ЗАДАНИЕ: Реализуйте прямой проход
        # 1. Встраивание (Embedding)
        # 2. LSTM
        # 3. Полносвязный слой
        pass

# Параметры модели
embed_size = 128
hidden_size = 256
num_layers = 2

# Создание модели
model = CharLSTM(vocab_size, embed_size, hidden_size, num_layers).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Количество параметров: {total_params:,}")
print(f"Размер словаря: {vocab_size}")
print(f"Размер встраивания: {embed_size}")
print(f"Скрытый размер: {hidden_size}")
print(f"Количество слоёв: {num_layers}")


In [ ]:
# ============================================================
# 5. ОБУЧЕНИЕ МОДЕЛИ
# ============================================================

# ЗАДАНИЕ: Реализуйте функцию train_model
def train_model(model, train_loader, vocab_size, num_epochs=20, lr=0.002):
    """
    Обучение модели генерации текста.
    
    Параметры:
    - model: модель (CharLSTM)
    - train_loader: DataLoader с обучающими данными
    - vocab_size: размер словаря
    - num_epochs: количество эпох
    - lr: скорость обучения
    
    Возвращает:
    - losses: список потерь на каждой эпохе
    - model: обученная модель
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []
    
    # ЗАДАНИЕ: Реализуйте цикл обучения
    # for epoch in range(num_epochs):
    #     model.train()
    #     epoch_loss = 0
    #     for batch_X, batch_y in train_loader:
    #         ...
    #         outputs, _ = model(batch_X)
    #         # Берём выход для последнего временного шага
    #         outputs = outputs[:, -1, :]  # (batch_size, vocab_size)
    #         loss = criterion(outputs, batch_y)
    #         ...
    #     losses.append(avg_loss)
    
    return losses, model

# Обучение модели
print("Начало обучения...")
losses, model = train_model(
    model, 
    train_loader, 
    vocab_size=vocab_size, 
    num_epochs=25, 
    lr=0.002
)
print("Обучение завершено!")

# Визуализация потерь
plt.figure(figsize=(12, 4))
plt.plot(losses, label='Потери на обучении', linewidth=2)
plt.xlabel('Эпоха')
plt.ylabel('Потери (CrossEntropy)')
plt.title('Динамика потерь при обучении')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# ============================================================
# 6. ФУНКЦИЯ ГЕНЕРАЦИИ ТЕКСТА С ТЕМПЕРАТУРОЙ
# ============================================================

# ЗАДАНИЕ: Реализуйте функцию generate_text
def generate_text(model, start_string, length, temperature=1.0, seq_length=100):
    """
    Генерация текста с использованием обученной модели.
    
    Параметры:
    - model: обученная модель
    - start_string: начальная строка (seed)
    - length: количество символов для генерации
    - temperature: температура для sampling
    - seq_length: длина последовательности
    
    Возвращает:
    - generated_text: сгенерированный текст
    """
    model.eval()
    
    # ЗАДАНИЕ: Реализуйте генерацию
    # 1. Преобразование начальной строки в индексы
    # 2. Цикл генерации символов
    # 3. Sampling с температурой
    # 4. Преобразование индексов в текст
    
    return generated_text

In [ ]:
# ============================================================
# 7. ГЕНЕРАЦИЯ ПРИМЕРОВ С РАЗНЫМИ ТЕМПЕРАТУРАМИ
# ============================================================

# Параметры генерации
start_string = "Я вас люблю"
length = 300
temperatures = [0.5, 1.0, 1.5]

print(f"Начальная строка: '{start_string}'")
print(f"Длина генерации: {length} символов")
print("=" * 70)

# ЗАДАНИЕ: Сгенерируйте тексты для каждой температуры
for T in temperatures:
    print(f"\n{'='*30} Температура: {T} {'='*30}")
    # generated = generate_text(...)
    # print(generated[:500])
    print("-" * 70)

In [ ]:
# ============================================================
# 8. АНАЛИЗ КАЧЕСТВА ГЕНЕРАЦИИ
# ============================================================

print("\n" + "="*70)
print("АНАЛИЗ КАЧЕСТВА ГЕНЕРАЦИИ")
print("="*70)

# ЗАДАНИЕ: Напишите анализ качества генерации
# - При T=0.5: ...
# - При T=1.0: ...
# - При T=1.5: ...

In [ ]:
# ============================================================
# 9. ДОПОЛНИТЕЛЬНЫЕ ЗАДАНИЯ
# ============================================================

# 9.1. Сравнение LSTM и GRU
# ЗАДАНИЕ: Реализуйте класс CharGRU, обучите модель GRU и сравните потери с LSTM



In [ ]:
# 9.2. Влияние длины последовательности
# ЗАДАНИЕ: Исследуйте влияние seq_length (50, 100, 200)

In [ ]:
# ============================================================
# 10. ВЫВОДЫ
# ============================================================

print("\n" + "="*70)
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("="*70)

print(f"""
Характеристики модели:
- Размер алфавита: {vocab_size} символов
- Длина последовательности: {seq_length}
- Размер встраивания: {embed_size}
- Скрытый размер LSTM: {hidden_size}
- Количество слоёв: {num_layers}
- Количество параметров: {total_params:,}

Качество обучения:
- Итоговые потери: {losses[-1]:.4f}

Рекомендации:
- Для осмысленной генерации используйте T ≈ 0.8–1.0
- Для стилизации (менее разнообразно) используйте T ≈ 0.5
- Для экспериментов можно использовать T > 1.2
""")

# Сохранение модели (опционально)
# torch.save(model.state_dict(), 'char_lstm_model.pth')
# print("Модель сохранена как 'char_lstm_model.pth'")
